In [3]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import joblib

# Create directories if they don't exist
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

def generate_synthetic_logs(num_rows=1000):
    """Generates synthetic cybersecurity assessment logs for training and testing."""
    np.random.seed(42)
    
    timestamps = pd.date_range(start="2026-10-01 00:00:00", periods=num_rows, freq="min")
    source_ips = [f"192.168.1.{np.random.randint(1, 50)}" for _ in range(num_rows)]
    
    # Inject some malicious IPs simulating attackers
    attacker_ips = ["10.0.0.99", "45.33.22.11", "185.199.108.153"]
    for i in range(int(num_rows * 0.05)):
        source_ips[np.random.randint(0, num_rows)] = np.random.choice(attacker_ips)
        
    # FIXED: Changed np.choice to np.random.choice
    event_types = np.random.choice(
        ["HTTP_GET", "HTTP_POST", "LOGIN_SUCCESS", "LOGIN_FAILURE", "PORT_SCAN", "SQL_INJECTION"],
        size=num_rows,
        p=[0.5, 0.2, 0.15, 0.1, 0.03, 0.02]
    )
    
    failure_counts = []
    bytes_transferred = []
    status_codes = []
    
    for et in event_types:
        if et == "LOGIN_FAILURE":
            failure_counts.append(np.random.randint(3, 15))
            bytes_transferred.append(np.random.randint(200, 500))
            status_codes.append(401)
        elif et in ["PORT_SCAN", "SQL_INJECTION"]:
            failure_counts.append(np.random.randint(0, 5))
            bytes_transferred.append(np.random.randint(5000, 50000))
            status_codes.append(403)
        else:
            failure_counts.append(0)
            bytes_transferred.append(np.random.randint(500, 4000))
            status_codes.append(200)

    df = pd.DataFrame({
        "timestamp": timestamps,
        "source_ip": source_ips,
        "event_type": event_types,
        "failure_count": failure_counts,
        "bytes_transferred": bytes_transferred,
        "status_code": status_codes
    })
    
    # Ground truth label for evaluation reference
    df["is_suspicious"] = df.apply(
        lambda row: 1 if row["event_type"] in ["PORT_SCAN", "SQL_INJECTION"] or row["failure_count"] > 5 else 0,
        axis=1
    )
    
    df.to_csv("data/raw_logs.csv", index=False)
    print("[+] Generated raw logs dataset at data/raw_logs.csv")
    return df

def train_model():
    """Trains an Isolation Forest anomaly detection model."""
    df = pd.read_csv("data/raw_logs.csv")
    
    # Feature engineering for ML
    features = df[["failure_count", "bytes_transferred", "status_code"]]
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(features)
    
    # Isolation Forest for unsupervised anomaly detection
    model = IsolationForest(contamination=0.08, random_state=42)
    model.fit(X_scaled)
    
    # Save artifacts
    joblib.dump(model, "models/isolation_forest.pkl")
    joblib.dump(scaler, "models/scaler.pkl")
    print("[+] Model trained and saved successfully to models/")

if __name__ == "__main__":
    generate_synthetic_logs()
    train_model()

[+] Generated raw logs dataset at data/raw_logs.csv
[+] Model trained and saved successfully to models/


In [4]:
import pandas as pd
from datetime import timedelta

def correlate_alerts(log_file="data/raw_logs.csv"):
    """Correlates security alerts based on timestamps, IPs, and event severity."""
    df = pd.read_csv(log_file)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    
    # Filter only suspicious or anomalous logs
    suspicious_df = df[(df["failure_count"] > 3) | (df["event_type"].isin(["PORT_SCAN", "SQL_INJECTION"]))]
    
    incidents = []
    grouped = suspicious_df.groupby("source_ip")
    
    for ip, group in grouped:
        incident_id = f"INC-{abs(hash(ip)) % 10000:04d}"
        event_count = len(group)
        start_time = group["timestamp"].min()
        end_time = group["timestamp"].max()
        event_types = group["event_type"].unique().tolist()
        
        # Determine risk severity
        if event_count > 5 or "SQL_INJECTION" in event_types:
            severity = "CRITICAL"
        elif event_count > 2:
            severity = "HIGH"
        else:
            severity = "MEDIUM"
            
        incidents.append({
            "incident_id": incident_id,
            "source_ip": ip,
            "severity": severity,
            "total_events": event_count,
            "event_types": event_types,
            "start_time": str(start_time),
            "end_time": str(end_time),
            "supporting_evidence": f"Detected {event_count} anomalous events originating from {ip} between {start_time} and {end_time}."
        })
        
    return incidents

if __name__ == "__main__":
    print(correlate_alerts())

[{'incident_id': 'INC-7513', 'source_ip': '10.0.0.99', 'severity': 'MEDIUM', 'total_events': 2, 'event_types': ['LOGIN_FAILURE'], 'start_time': '2026-10-01 04:55:00', 'end_time': '2026-10-01 09:40:00', 'supporting_evidence': 'Detected 2 anomalous events originating from 10.0.0.99 between 2026-10-01 04:55:00 and 2026-10-01 09:40:00.'}, {'incident_id': 'INC-5600', 'source_ip': '185.199.108.153', 'severity': 'CRITICAL', 'total_events': 1, 'event_types': ['SQL_INJECTION'], 'start_time': '2026-10-01 09:06:00', 'end_time': '2026-10-01 09:06:00', 'supporting_evidence': 'Detected 1 anomalous events originating from 185.199.108.153 between 2026-10-01 09:06:00 and 2026-10-01 09:06:00.'}, {'incident_id': 'INC-7267', 'source_ip': '192.168.1.1', 'severity': 'MEDIUM', 'total_events': 1, 'event_types': ['LOGIN_FAILURE'], 'start_time': '2026-10-01 02:17:00', 'end_time': '2026-10-01 02:17:00', 'supporting_evidence': 'Detected 1 anomalous events originating from 192.168.1.1 between 2026-10-01 02:17:00 a

In [6]:
def map_to_compliance(event_types):
    """Maps detected security events to NIST CSF and ISO 27001 controls."""
    mappings = []
    
    control_database = {
        "LOGIN_FAILURE": {
            "nist_csf": "PR.AC-1 (Access Control Management)",
            "iso_27001": "A.9.4.2 (Secure Log-on Procedures)",
            "remediation": "Enforce account lockout policies and IP rate-limiting after 5 failed attempts."
        },
        "PORT_SCAN": {
            "nist_csf": "DE.CM-1 (Network Monitoring for Anomalies)",
            "iso_27001": "A.13.1.1 (Network Controls)",
            "remediation": "Block source IP via firewall rules and review perimeter intrusion detection systems."
        },
        "SQL_INJECTION": {
            "nist_csf": "PR.DS-5 (Data Protection - Integrity)",
            "iso_27001": "A.14.1.2 (Securing Application Services)",
            "remediation": "Implement parameterized queries, input sanitization, and Web Application Firewall (WAF) filtering."
        }
    }
    
    for et in event_types:
        if et in control_database:
            mappings.append({
                "event_type": et,
                **control_database[et]
            })
        else:
            mappings.append({
                "event_type": et,
                "nist_csf": "ID.RA-1 (Vulnerability Identification)",
                "iso_27001": "A.12.6.1 (Management of Technical Vulnerabilities)",
                "remediation": "Perform routine log inspection and patch underlying software assets."
            })
            
    return mappings